# Evaluation Metrics in LLM and RAG Systems

---

## 1. Traditional Metrics: Precision, Recall, F1-Score

In classical Machine Learning (e.g., classification tasks), we often rely on **Precision, Recall, and F1-score** as the core evaluation metrics.

- **Precision**: Out of the predicted positives, how many are correct?  
  *Example*: A RAG system retrieved 10 passages, 6 are relevant.  
  Precision = 6/10 = **0.6**

- **Recall**: Out of all relevant documents, how many did we retrieve?  
  *Example*: There were 8 relevant passages in the corpus, we retrieved 6.  
  Recall = 6/8 = **0.75**

- **F1-Score**: The harmonic mean of precision and recall, balancing the two.  
  *Example*: Using the above numbers,  
  F1 = 2 × (0.6 × 0.75) / (0.6 + 0.75) ≈ **0.67**

---

### ❌ Why Precision/Recall/F1 Are Not Enough?

Precision, Recall, and F1-score are excellent for document retrieval tasks but **fall short for LLM and RAG-based systems**, where both retrieval quality and answer generation quality matter.  
They fail to capture dimensions that go beyond simple document relevance:

#### 🔹 Faithfulness (Groundedness to Retrieved Context)
- **What it means**: Does the generated answer stay true to the retrieved context, or does it hallucinate information?  
- **Why they fail**: They only check retrieval, not whether the answer used them correctly.  
- *Example*:  
  - Retrieved doc: *“The capital of France is Paris.”*  
  - Generated answer: *“The capital of France is Lyon.”*  
  - Retrieval scores → High  
  - Actual answer → **Unfaithful / hallucinated**

#### 🔹 Relevance of the Final Response
- **What it means**: Does the final generated response actually answer the user’s query directly and completely?  
- **Why they fail**: They stop at retrieval and don’t measure final answer quality.  
- *Example*:  
  - Query: *“What year was the iPhone first released?”*  
  - Retrieved doc: *“Apple released the first iPhone in 2007.”*  
  - Answer A: *“The iPhone was first released in 2007.”* → ✅ Relevant  
  - Answer B: *“Apple is a famous tech company.”* → ❌ Not relevant  
  - Precision/Recall → Same for both cases, but only Answer A is useful.

#### 🔹 Conversational Dynamics (Multi-turn Interaction)
- **What it means**: In multi-turn conversations, the model must track context, follow its role, and stay coherent.  
- **Why they fail**: They are single-turn metrics and cannot evaluate flow.  
- *Example*:  
  - User (turn 1): *“Who is the Prime Minister of India?”*  
    Model: *“Narendra Modi is the Prime Minister of India.”*  
  - User (turn 2): *“What was his role before becoming PM?”*  
    - Correct answer: *“He served as the Chief Minister of Gujarat from 2001 to 2014.”*  
    - Incorrect: *“I don’t know who you mean by ‘he’.”*  
  - Precision/Recall/F1 → Cannot score this.  

➡️ This is where **advanced evaluation frameworks like RAGAS and G-Eval** step in.

---

## 2. RAG-Specific Metrics

Retrieval-Augmented Generation (RAG) involves two steps:
1. **Retrieving documents (Information Retrieval)**
2. **Generating an answer using those documents (Response Generation)**

So, evaluation must cover both retrieval quality and answer quality.

---

### 🔹 Contextual Precision
**Definition**: Measures how many of the retrieved contexts are actually relevant to the query.

\[
\text{Contextual Precision} = \frac{\text{Relevant Retrieved Contexts}}{\text{Total Retrieved Contexts}}
\]

- **Factors affecting score**:
  - Size of retrieval set (large set lowers precision).
  - Noise in pipeline (irrelevant passages).
  - Embedding/keyword quality.

- **Example**:  
  Query: *“Who discovered penicillin?”*  
  Retrieved 5 passages → 4 relevant, 1 irrelevant.  
  Precision = 4/5 = **0.8**

---

### 🔹 Contextual Recall
**Definition**: Measures how many of the total relevant contexts were retrieved.

\[
\text{Contextual Recall} = \frac{\text{Relevant Retrieved Contexts}}{\text{Total Relevant Contexts in Corpus}}
\]

- **Factors affecting score**:
  - Small top-k lowers recall.
  - Missing semantically close docs.
  - Poor query expansion.

- **Example**:  
  10 relevant passages exist, system retrieved 4.  
  Recall = 4/10 = **0.4**

---

### 🔹 Contextual Relevancy
**Definition**: Evaluates whether retrieved docs are semantically useful, not just keyword overlap.

- **Factors**:
  - Semantic alignment.
  - Actual support for answering.

- **Example**:  
  Query: *“Who discovered penicillin?”*  
  - Doc A: *“Alexander Fleming discovered penicillin in 1928.”* → ✅ Relevant  
  - Doc B: *“Penicillin is an antibiotic.”* → ❌ Low relevance  
  Relevancy = 1/2 = **0.5**

---

### 🔹 Answer Relevancy
**Definition**: Checks whether the generated answer addresses the user query.

\[
\text{Answer Relevancy} = \frac{\text{Relevant Answer Segments}}{\text{Total Answer Segments}}
\]

- **Factors**:
  - Completeness.  
  - Staying on-topic.  
  - Conciseness.  

- **Example**:  
  - Query: *“Capital of France?”*  
  - Answer: *“Paris is the capital of France.”* → ✅ Relevant (1)  
  - Answer: *“France is in Europe.”* → ❌ Not relevant (0)  
  Avg. relevancy = **0.5**

---

### 🔹 Faithfulness
**Definition**: Whether the generated answer is factually consistent with retrieved context.

- **Factors**:
  - Hallucination.  
  - Misinterpretation.  
  - Over-generalization.  

- **Example**:  
  - Context: *“The capital of France is Paris.”*  
  - Answer A: *“The capital of France is Lyon.”* → ❌ Unfaithful (0)  
  - Answer B: *“Paris is the capital of France.”* → ✅ Faithful (1)

---

## 3. RAGAS (Retrieval-Augmented Generation Assessment)

RAGAS provides a **unified framework** by combining retrieval + generation metrics.  
It focuses on **semantic evaluation**, not just exact matches.  
It uses **LLM-based judges** to score:
- Contextual Precision  
- Contextual Recall  
- Answer Relevance  
- Faithfulness  

**Example**:  
- Query: *“When was the iPhone first released?”*  
- Context: *“Apple launched the first iPhone in 2007.”*  
- Answer: *“The iPhone was first released in 2007 by Apple.”*  

✅ High contextual precision  
✅ High faithfulness  
✅ High answer relevancy  

---

## 4. G-Eval: LLM-as-a-Judge

- Uses **LLM as a judge** with structured prompts.  
- Great for open-ended answers where BLEU/ROUGE fail.  
- Evaluates: **fluency, coherence, factuality, style**.  

*Example*: Summarization task – instead of exact match, G-Eval checks if **all key points are covered**.

---

## 5. DAG (Directed Acyclic Graph) for Multi-Metric Evaluation

- **Nodes** = metrics (precision, recall, faithfulness, etc.)  
- **Edges** = dependencies (e.g., faithfulness depends on retrieval quality).  
- Ensures logical flow from retrieval → reasoning → generation.

---

## 6. Agentic Metrics (for Agents and Tool Use)

When LLMs act as **agents** (tools, APIs, multi-step reasoning):

- **Task Completion**: Did the agent achieve the end goal?  
  *Example*: Successfully booked a flight.  
- **Tool Correctness**: Was the right tool/API used correctly?  
  *Example*: Used weather API vs mistakenly calling calendar API.

---

## 7. Other Critical Metrics

- **Hallucination**: Checks fabricated info.  
- **Summarization Quality**: Are key points captured?  
- **Bias**: Detects unintended bias (gender, cultural, political).  
- **Toxicity**: Detects unsafe or offensive output.

---

## 8. Conversational Metrics (for Multi-turn Chat)

- **Knowledge Retention**: Remembers past turns.  
- **Conversation Completeness**: Covers all aspects of query.  
- **Conversation Relevancy**: Stays on-topic.  
- **Role Adherence**: Follows its assigned role.  

*Example*:  
- User: *“Explain Newton’s first law.”*  
  - Assistant: *“Newton’s first law states that…”* → ✅ Role adhered  
  - Assistant: *“I don’t want to talk about physics.”* → ❌ Role violation  

---

## ✅ Final Takeaway

- Precision, Recall, F1 are **necessary but insufficient** for LLM/RAG evaluation.  
- **RAGAS** → semantic evaluation (retrieval + generation).  
- **G-Eval** → LLM-as-judge for nuanced outputs.  
- **Agentic & Conversational metrics** → real-world scenarios.  

A **holistic evaluation** must combine:
- Traditional IR metrics (Precision, Recall, F1)  
- RAG metrics (Faithfulness, Contextual Precision/Recall, Relevance)  
- Agentic & Conversational metrics (Task Completion, Role Adherence, Hallucination, Bias, etc.)  

👉 Only then can we **trust LLMs and RAG systems in production**.


##DAG EVALUATION

In [1]:
import math
import statistics

# --------------------------
# Example qrels (ground truth)
# --------------------------
# Binary relevance for two queries, graded relevance for one query (0,1,2).
qrels = {
    'Sweet pastry': {
        'Donut': 1, 'Muffin': 1, 'Scone': 1
    },
    'Suitable for lunch': {
        'Sandwich': 1, 'Bagel': 1, 'Roll': 1, 'Pretzel': 1
    },
    # graded relevance (2 = highly relevant, 1 = relevant, 0 = not)
    'Goes well with jam': {
        'Bagel': 2, 'Croissant': 2, 'Roll': 2,
        'Scone': 1, 'Muffin': 1, 'Donut': 1
    },
}

# --------------------------
# Example system runs (ranked lists)
# --------------------------
# Each list is ranked: index 0 = rank 1 (best), index 1 = rank 2, ...
runs = {
    'Sweet pastry': ['Croissant', 'Donut', 'Scone', 'Sandwich', 'Muffin'],
    'Suitable for lunch': ['Muffin', 'Donut', 'Sandwich', 'Bagel', 'Roll'],
    'Goes well with jam': ['Pretzel', 'Bagel', 'Muffin', 'Donut', 'Croissant'],
}

# --------------------------
# Metric helpers
# --------------------------
def precision_at_k(retrieved, qrel, k):
    rels = [1 if qrel.get(doc, 0) > 0 else 0 for doc in retrieved[:k]]
    return sum(rels) / k

def recall_at_k(retrieved, qrel, k):
    rels = [1 if qrel.get(doc, 0) > 0 else 0 for doc in retrieved[:k]]
    total_relevant = sum(1 for v in qrel.values() if v > 0)
    return sum(rels) / total_relevant if total_relevant > 0 else 0.0

# Average Precision @ k (standard definition): sum_{i<=k} P(i)*rel(i) / (total_relevant)
def average_precision_at_k(retrieved, qrel, k):
    rels = [1 if qrel.get(doc, 0) > 0 else 0 for doc in retrieved[:k]]
    precisions_at_relevant = []
    num_relevant_found = 0
    for i, r in enumerate(rels, start=1):
        if r == 1:
            num_relevant_found += 1
            precisions_at_relevant.append(num_relevant_found / i)
    total_relevant = sum(1 for v in qrel.values() if v > 0)
    return sum(precisions_at_relevant) / total_relevant if total_relevant > 0 else 0.0

# DCG / IDCG / NDCG for graded relevance
def dcg_at_k(retrieved, qrel, k):
    rels = [qrel.get(doc, 0) for doc in retrieved[:k]]
    dcg = 0.0
    for i, rel in enumerate(rels, start=1):
        dcg += (2**rel - 1) / math.log2(i + 1)
    return dcg

def idcg_at_k(qrel, k):
    ideal_rels = sorted([v for v in qrel.values() if v > 0], reverse=True)[:k]
    idcg = 0.0
    for i, rel in enumerate(ideal_rels, start=1):
        idcg += (2**rel - 1) / math.log2(i + 1)
    return idcg

def ndcg_at_k(retrieved, qrel, k):
    idcg = idcg_at_k(qrel, k)
    if idcg == 0:
        return 0.0
    return dcg_at_k(retrieved, qrel, k) / idcg

# --------------------------
# Run evaluation (per-query + aggregated)
# --------------------------
def evaluate_all(runs, qrels, k=4):
    per_query = {}
    for q in runs:
        p = precision_at_k(runs[q], qrels[q], k)
        r = recall_at_k(runs[q], qrels[q], k)
        ap = average_precision_at_k(runs[q], qrels[q], k)
        ndcg = ndcg_at_k(runs[q], qrels[q], k)
        per_query[q] = {'P@{}'.format(k): p, 'R@{}'.format(k): r, 'AP@{}'.format(k): ap, 'NDCG@{}'.format(k): ndcg}
    # overall MAP and mean NDCG
    map_k = statistics.mean([v['AP@{}'.format(k)] for v in per_query.values()])
    mean_ndcg = statistics.mean([v['NDCG@{}'.format(k)] for v in per_query.values()])
    return per_query, map_k, mean_ndcg

# Choose k (top-k). In a class demo you can change this to 4,5,10...
k = 4
per_query, map_k, mean_ndcg = evaluate_all(runs, qrels, k)

# Pretty print
for q, metrics in per_query.items():
    print(f"Query: {q}")
    for name, val in metrics.items():
        print(f"  {name}: {val:.6f}")
print(f"\nMAP@{k}: {map_k:.6f}")
print(f"Mean NDCG@{k}: {mean_ndcg:.6f}")


Query: Sweet pastry
  P@4: 0.500000
  R@4: 0.666667
  AP@4: 0.388889
  NDCG@4: 0.530721
Query: Suitable for lunch
  P@4: 0.500000
  R@4: 0.500000
  AP@4: 0.208333
  NDCG@4: 0.363318
Query: Goes well with jam
  P@4: 0.750000
  R@4: 0.500000
  AP@4: 0.319444
  NDCG@4: 0.413788

MAP@4: 0.305556
Mean NDCG@4: 0.435942
